In [1]:
import pandas as pd

# load all stair-relevant attribute train files
train_files = {
    "fall_height": pd.read_csv("../data/processed/train/attr_fall_height_train.csv"),
    "has_pedestrian_railing": pd.read_csv("../data/processed/train/attr_has_pedestrian_railing_train.csv"),
    "material": pd.read_csv("../data/processed/train/attr_material_frame_tank_body_train.csv"),
    "number_of_steps": pd.read_csv("../data/processed/train/attr_number_of_steps_train.csv"),
    "structure_position": pd.read_csv("../data/processed/train/attr_structure_position_train.csv"),
}

# filter each to stairs only
stair_assets = {}
for attr, df in train_files.items():
    stair_assets[attr] = set(
        df[df["profile_name"] == "Stairs"]["asset_id"].unique()
    )

# find overlap across all 5
overlap = set.intersection(*stair_assets.values())
print(f"Assets in all 5 stair attribute train files: {len(overlap)}")
print(list(overlap)[:5])

Assets in all 5 stair attribute train files: 3
[np.int64(126777), np.int64(121499), np.int64(121503)]


In [2]:
test_asset_id = 126777

# verify it exists in all files
for attr, df in train_files.items():
    row = df[df["asset_id"] == test_asset_id]
    print(f"{attr}: {row[f'attr_{attr}' if attr != 'material' else 'attr_material_frame,_tank,_body'].values}")

fall_height: [0.5]
has_pedestrian_railing: ['1 railing']
material: ['Timber/Wood']
number_of_steps: [12.]
structure_position: ['Elevated']


In [3]:
# use structure position train file since it has all columns including image_path
test_df = train_files["structure_position"][
    train_files["structure_position"]["asset_id"] == test_asset_id
]

print(f"Images for asset {test_asset_id}: {len(test_df)}")
test_df[["asset_id", "profile_name", "image_path"]].to_csv(
    "../data/processed/train/pipeline_test_asset.csv", index=False
)

Images for asset 126777: 1


In [ ]:
#comparing test with ground truth labels

train_files = {
    "attr_fall_height": pd.read_csv("../data/processed/train/attr_fall_height_train.csv"),
    "attr_has_pedestrian_railing": pd.read_csv("../data/processed/train/attr_has_pedestrian_railing_train.csv"),
    "attr_material_frame,_tank,_body": pd.read_csv("../data/processed/train/attr_material_frame_tank_body_train.csv"),
    "attr_number_of_steps": pd.read_csv("../data/processed/train/attr_number_of_steps_train.csv"),
    "attr_structure_position": pd.read_csv("../data/processed/train/attr_structure_position_train.csv"),
}

for attr, df in train_files.items():
    row = df[df["asset_id"] == 126777]
    if len(row) > 0:
        print(f"{attr}: {row[attr].values[0]}")

attr_fall_height: 0.5
attr_has_pedestrian_railing: 1 railing
attr_material_frame,_tank,_body: Timber/Wood
attr_number_of_steps: 12.0
attr_structure_position: Elevated


In [10]:
fall_height_bin_train = pd.read_csv("../data/processed/train/fall_height_bin_train.csv")
steps_bin_train = pd.read_csv("../data/processed/train/steps_bin_train.csv")

print("fall_height_bin - asset 126777 in train:", 
      126777 in fall_height_bin_train["asset_id"].values)

print("steps_bin - asset 126777 in train:", 
      126777 in steps_bin_train["asset_id"].values)

fall_height_bin - asset 126777 in train: True
steps_bin - asset 126777 in train: True


In [11]:
print(fall_height_bin_train[fall_height_bin_train["asset_id"] == 126777][["asset_id", "fall_height_bin"]])
print(steps_bin_train[steps_bin_train["asset_id"] == 126777][["asset_id", "steps_bin"]])

    asset_id    fall_height_bin
67    126777  medium (0.5-1.2m)
    asset_id       steps_bin
37    126777  medium (10-20)


In [13]:
import pandas as pd
preds = pd.read_csv("../results/test_pipeline_stairs.csv")
print(preds.columns.tolist())
print(preds.iloc[0])

['asset_id', 'timestamp', 'model', 'response', 'fall_height_bin_value', 'fall_height_bin_confidence', 'has_pedestrian_railing_value', 'has_pedestrian_railing_confidence', 'material_frame_tank_body_value', 'material_frame_tank_body_confidence', 'steps_bin_value', 'steps_bin_confidence', 'structure_position_value', 'structure_position_confidence']
asset_id                                                                          126777
timestamp                                                     2026-05-19T21:27:46.851836
model                                                             gemini-3-flash-preview
response                               {\n    "fall_height": {\n    "value": "high (>...
fall_height_bin_value                                                       high (>1.2m)
fall_height_bin_confidence                                                           0.9
has_pedestrian_railing_value                                                   1 railing
has_pedestrian_railing_confid

In [6]:
PROMPT_TEMPLATE = """
    You are an expert in park infrastructure analysis.

    Using ALL provided images of this single stair asset, identify the most likely
    attribute values. For each of the following attributes, the possible values are
    given below. Predict exactly ONE value from the listed options for each
    attribute, and provide a confidence score (0.0-1.0) for each prediction.

    Attributes to predict:
    - fall_height: low (<0.5m) | medium (0.5m-1.2m) | high (>1.2m)
    - has_pedestrian_railing: 2 railings | 1 railing | no railings
    - material_frame_tank_body: PVC | Gravel | Natural Surface | Earth-filled |
                                Aluminum | Metal | Steel | Rock/Stone | Concrete |
                                Box Step | Timber/Wood
    - number_of_steps: few (<10) | medium (10-20) | many (>20)
    - structure_position: Elevated | At-Grade | Other

    Return ONLY a valid JSON object with this exact schema (no markdown, no prose):
    {
        "<attribute_key>": {
        "value": "<predicted value or 'unable to determine'>",
        "confidence": <float 0.0-1.0>
        }
    }

    If you cannot determine an attribute from the images, set value to
    "unable to determine" and confidence to 0.0.
    """

In [9]:
import sys
from pathlib import Path

# add project root to path
sys.path.insert(0, str(Path("..").resolve()))

from src.vlm.predictors import predict_asset_attributes
import pandas as pd

df = pd.read_csv("../data/processed/train/pipeline_test_asset.csv")

result = predict_asset_attributes(
    asset_id=126777,
    df=df,
    model_name="gemini-3-flash-preview",
    prompt=PROMPT_TEMPLATE
)

print(result)

{'asset_id': 126777, 'error': "503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}", 'response': None}


In [14]:
import pandas as pd

train_only = pd.read_csv("../data/processed/train/train_only_assets.csv")
length_train = pd.read_csv("../data/processed/train/attr_length_train.csv")

# how many train_only assets have a length label
overlap = train_only[train_only["asset_id"].isin(length_train["asset_id"])]
print(f"Assets with length label in train_only: {len(overlap['asset_id'].unique())}")

Assets with length label in train_only: 819


In [17]:
import pandas as pd

train_only = pd.read_csv("../data/processed/train/train_only_assets.csv")
length_train = pd.read_csv("../data/processed/train/attr_length_train.csv")

# filter train_only to only assets with length label
length_assets = train_only[train_only["asset_id"].isin(length_train["asset_id"])]
length_assets.to_csv("../data/processed/train/train_only_length_assets.csv", index=False)

print(f"Assets with length label: {length_assets['asset_id'].nunique()}")

Assets with length label: 819


In [18]:
import pandas as pd

train_only = pd.read_csv("../data/processed/train/train_only_assets.csv")

print(f"Total rows: {len(train_only)}")
print(f"Unique asset IDs: {train_only['asset_id'].nunique()}")
print(f"Rows per asset:")
print(train_only.groupby('asset_id').size().describe())

# show assets with more than 1 row
duplicates = train_only[train_only.duplicated('asset_id', keep=False)]
print(f"\nAssets with multiple rows: {duplicates['asset_id'].nunique()}")
print(duplicates[['asset_id', 'profile_name', 'image_path']].head(10))

Total rows: 2582
Unique asset IDs: 1687
Rows per asset:
count    1687.000000
mean        1.530528
std         1.094085
min         1.000000
25%         1.000000
50%         1.000000
75%         2.000000
max        10.000000
dtype: float64

Assets with multiple rows: 517
    asset_id           profile_name  \
0      47664  Boardwalk < 1.2m High   
2      47664  Boardwalk < 1.2m High   
3      48119  Boardwalk < 1.2m High   
4      47664  Boardwalk < 1.2m High   
5      48119  Boardwalk < 1.2m High   
6      48119  Boardwalk < 1.2m High   
9      48934  Boardwalk < 1.2m High   
11     48934  Boardwalk < 1.2m High   
14     51547  Boardwalk < 1.2m High   
15     51547  Boardwalk < 1.2m High   

                                           image_path  
0   data/citywide/images/337/47664/86079__AST_EX_2...  
2   data/citywide/images/337/47664/86081__AST_EX_2...  
3   data/citywide/images/337/48119/13815__MI_20210...  
4   data/citywide/images/337/47664/86084__AST_EX_2...  
5   data/citywide/i

In [19]:
# check if attribute columns are identical across duplicate rows
length_train = pd.read_csv("../data/processed/train/attr_length_train.csv")

# get assets with multiple rows
multi_image_assets = train_only[train_only.duplicated('asset_id', keep=False)]['asset_id'].unique()

# check if labels are consistent across rows for same asset
inconsistent = []
for asset_id in multi_image_assets:
    rows = length_train[length_train['asset_id'] == asset_id]
    if len(rows) > 1:
        # check if length_bin is the same across all rows
        if rows['length_bin'].nunique() > 1:
            inconsistent.append(asset_id)

print(f"Assets with inconsistent length_bin labels: {len(inconsistent)}")
if len(inconsistent) > 0:
    print(inconsistent)
else:
    print("✅ All duplicate assets have consistent labels across rows")

Assets with inconsistent length_bin labels: 0
✅ All duplicate assets have consistent labels across rows


In [20]:
import pandas as pd
import glob

batches = glob.glob("../results/vlm_length_*.csv")
combined = pd.concat([pd.read_csv(f) for f in sorted(batches)])
combined = combined.drop_duplicates("asset_id")
combined.to_csv("../results/vlm_length_combined.csv", index=False)
print(f"Total assets: {len(combined)}")

Total assets: 70
